# General

For more informations, si the documentation *LLM and GenAI*.

# Import & Configs

In [1]:
from ollama import chat
import ollama
import time

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
%load_ext autoreload
%autoreload 2

from src.retrieval.retriever import MedicalRetriever

# Ollama Tests

In [21]:
"""
for chunk in ollama.chat(
    model="qwen2.5:1.5b",
    messages=[
        {
            "role": "user",
            "content": "What is glioblastoma?"
        }
    ],
    stream=True
):
    print(
        chunk["message"]["content"],
        end="",
        flush=True
    )
"""

'\nfor chunk in ollama.chat(\n    model="qwen2.5:1.5b",\n    messages=[\n        {\n            "role": "user",\n            "content": "What is glioblastoma?"\n        }\n    ],\n    stream=True\n):\n    print(\n        chunk["message"]["content"],\n        end="",\n        flush=True\n    )\n'

In [22]:
"""
start = time.time()

response = ollama.chat(
    model="qwen2.5:1.5b",
    messages=[
        {
            "role": "user",
            "content": "What is glioblastoma?"
        }
    ],
    options={
        "temperature": 0.1,
        "num_predict": 150
    }
)

elapsed = time.time() - start

print(f"{elapsed:.1f} seconds")
print(response["message"]["content"][:500])
"""

'\nstart = time.time()\n\nresponse = ollama.chat(\n    model="qwen2.5:1.5b",\n    messages=[\n        {\n            "role": "user",\n            "content": "What is glioblastoma?"\n        }\n    ],\n    options={\n        "temperature": 0.1,\n        "num_predict": 150\n    }\n)\n\nelapsed = time.time() - start\n\nprint(f"{elapsed:.1f} seconds")\nprint(response["message"]["content"][:500])\n'

Following an evaluation and benchmarking, it appears that "ollama.chat" takes a long time to start in Python. This may be due to a lack of compatibility with Python 3.12. For now, the stream mode seems to be more efficient.

# Load Retriever

In [5]:
retriever = MedicalRetriever()

/home/jeremy/Documents/dev/LLM_RAG/Medical_assistant/.ma_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
results = retriever.retrieve(
    "glioblastoma prognosis"
)

len(results["documents"][0])

Original query: glioblastoma prognosis
Processed query: glioblastoma prognosis


3

# Context Build

In [45]:
def build_context(results):

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    sections = []

    for doc, metadata in zip(documents, metadatas):

        sections.append(
            f"[PMID:{metadata['pmid']}]\n{doc}"
        )

    return "\n\n".join(sections)

Test:

In [46]:
context = build_context(results)

print(context[:1000])

[PMID:42189415]
Peripheral hematological landscapes as biomarkers for detecting postoperative progression in glioblastoma multiforme: a multivariable risk scoring approach. PURPOSE: Glioblastoma multiforme (GBM) is the most common malignant tumor with poor prognosis despite standard treatment. While various hematological parameters are prognostic for GBM survival, their potential in disease monitoring remains underexplored.

[PMID:41890862]
BACKGROUND: Glioblastomas (GBM) are highly aggressive, treatment-resistant brain tumors lacking clinically actionable, noninvasive prognostic biomarkers. Tumor response after standard-of-care chemoradiation (CRT) is difficult to interpret on imaging, and post-CRT MRI changes have not been well linked to molecular features or potential biomarkers.

[PMID:42131752]
In the present study, several quality indicators were evaluated in routine clinical care. METHODS: The EORTC Brain Tumor Group (BTG) developed quality indicators from published guidelines a

# Prompt Build

In [41]:
# To restrictive
def build_prompt(question, context):

    prompt = f"""
You are a medical assistant.

Answer ONLY using the provided context.

If the answer is not present in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

In [42]:
def build_prompt(question, context):

    prompt = f"""
You are a medical assistant.

Use the context below to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

Test:

In [30]:
question = "How is glioblastoma prognosis evaluated?"

results = retriever.retrieve(
    query=question
)

context = build_context(results)

prompt = build_prompt(
    question,
    context
)

print(prompt[:3000])

Original query: How is glioblastoma prognosis evaluated?
Processed query: glioblastoma prognosis evaluated

You are a medical assistant.

Use the context below to answer the question.

Context:
Peripheral hematological landscapes as biomarkers for detecting postoperative progression in glioblastoma multiforme: a multivariable risk scoring approach. PURPOSE: Glioblastoma multiforme (GBM) is the most common malignant tumor with poor prognosis despite standard treatment. While various hematological parameters are prognostic for GBM survival, their potential in disease monitoring remains underexplored.

BACKGROUND: Glioblastomas (GBM) are highly aggressive, treatment-resistant brain tumors lacking clinically actionable, noninvasive prognostic biomarkers. Tumor response after standard-of-care chemoradiation (CRT) is difficult to interpret on imaging, and post-CRT MRI changes have not been well linked to molecular features or potential biomarkers.

In the present study, several quality indic

# Generate Answer With Ollama

In [31]:
def generate_answer(prompt):

    response = ollama.chat(

        model="qwen2.5:1.5b",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        options={
            "temperature": 0.1,
            "num_predict": 150
        }
    )

    return response["message"]["content"]

In [33]:
def generate_answer(prompt):

    full_answer = ""

    for chunk in ollama.chat(
        model="qwen2.5:1.5b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.1,
            "num_predict": 150
        },
        stream=True
    ):

        token = chunk["message"]["content"]

        print(token, end="", flush=True)

        full_answer += token

    return full_answer

In [47]:
question = "How is glioblastoma prognosis evaluated?"

results = retriever.retrieve(
    query=question
)

context = build_context(results)

prompt = build_prompt(
    question,
    context
)

print(f"Prompt:\n{prompt}")
print("\n")
start = time.time()
answer = generate_answer(prompt)
elapsed = time.time() - start

print(f"{elapsed:.1f} seconds")
#print(answer)

Original query: How is glioblastoma prognosis evaluated?
Processed query: glioblastoma prognosis evaluated
Prompt:

You are a medical assistant.

Use the context below to answer the question.

Context:
[PMID:42189415]
Peripheral hematological landscapes as biomarkers for detecting postoperative progression in glioblastoma multiforme: a multivariable risk scoring approach. PURPOSE: Glioblastoma multiforme (GBM) is the most common malignant tumor with poor prognosis despite standard treatment. While various hematological parameters are prognostic for GBM survival, their potential in disease monitoring remains underexplored.

[PMID:41890862]
BACKGROUND: Glioblastomas (GBM) are highly aggressive, treatment-resistant brain tumors lacking clinically actionable, noninvasive prognostic biomarkers. Tumor response after standard-of-care chemoradiation (CRT) is difficult to interpret on imaging, and post-CRT MRI changes have not been well linked to molecular features or potential biomarkers.

[PM

# Add Sources

In [38]:
def display_sources(results):

    print("\nSources used:\n")

    for i, metadata in enumerate(
        results["metadatas"][0],
        start=1
    ):

        print(
            f"[{i}] {metadata['title']} "
            f"({metadata['year']}) "
            f"PMID:{metadata['pmid']}"
        )

In [39]:
results = retriever.retrieve(
    query="glioblastoma prognosis"
)

Original query: glioblastoma prognosis
Processed query: glioblastoma prognosis


In [40]:
display_sources(results)


Sources used:

[1] Peripheral hematological landscapes as biomarkers for detecting postoperative progression in glioblastoma multiforme: a multivariable risk scoring approach (2026) PMID:42189415
[2] Temporal Integration of Serum Proteomics, Metabolomics and MRI Tumor Volumetrics via Deep Learning Identifies Systemic Mediators of Glioblastoma Response to Chemoradiotherapy (2026) PMID:41890862
[3] Prediction of treatment failure in patients with glioblastoma with perfusion MRI and molecular biomarkers (2026) PMID:42200192


# Pipeline

In [48]:
def ask_medical_assistant(
    question,
    retriever,
    debug=False
):

    results = retriever.retrieve(
        query=question
    )
    
    context = build_context(results)
    
    prompt = build_prompt(
        question,
        context
    )

    if debug:
        print(f"Prompt:\n{prompt}")
        print("\n")
        start = time.time()
    
    answer = generate_answer(prompt)
    
    if debug:
        elapsed = time.time() - start
        print(f"{elapsed:.1f} seconds")

    display_sources(results)
    
    return {
        "question": question,
        "context": context,
        "answer": answer
    }

## Test

In [49]:
response = ask_medical_assistant(

    "How is glioblastoma prognosis evaluated?",

    retriever
)

Original query: How is glioblastoma prognosis evaluated?
Processed query: glioblastoma prognosis evaluated
Glioblastoma prognosis is evaluated through various methods including:

- Hematological parameters: While not explicitly mentioned in the provided context, peripheral hematological landscapes have been explored as potential biomarkers for detecting postoperative progression in GBM. This suggests that changes in blood cell counts or other hematology indicators might be used to monitor disease progression.

- Tumor response after standard-of-care chemoradiation (CRT): The study notes that tumor response is difficult to interpret on imaging, indicating that MRI and other imaging techniques may not provide clear information about the effectiveness of treatment. This implies that post-CRT MRI changes have not been well linked to molecular features or potential biomarkers.

- Quality indicators: The context mentions that quality indicators were evaluated in routine
Sources used:

[1] Pe

In [50]:
print(
    response["answer"]
)

Glioblastoma prognosis is evaluated through various methods including:

- Hematological parameters: While not explicitly mentioned in the provided context, peripheral hematological landscapes have been explored as potential biomarkers for detecting postoperative progression in GBM. This suggests that changes in blood cell counts or other hematology indicators might be used to monitor disease progression.

- Tumor response after standard-of-care chemoradiation (CRT): The study notes that tumor response is difficult to interpret on imaging, indicating that MRI and other imaging techniques may not provide clear information about the effectiveness of treatment. This implies that post-CRT MRI changes have not been well linked to molecular features or potential biomarkers.

- Quality indicators: The context mentions that quality indicators were evaluated in routine
